In [29]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from pathlib import Path

In [30]:
project_path = Path("/Users/irinafendley/Projects/Loan_Default")

df = pd.read_csv(
    project_path / "data/processed/loan_clean.csv"
)

In [31]:
model = joblib.load(
    project_path / "data/models/logistic_regression.pkl"
)

In [32]:
print(model.feature_names_in_)

['Age' 'Income' 'LoanAmount' 'CreditScore' 'MonthsEmployed'
 'NumCreditLines' 'InterestRate' 'LoanTerm' 'DTIRatio' 'HasMortgage'
 'HasDependents' 'HasCoSigner' 'Income_to_Loan' 'Debt_burden'
 'Score_pressure' 'Risk_rate' 'Risk_score' 'Education_High School'
 "Education_Master's" 'Education_PhD' 'EmploymentType_Part-time'
 'EmploymentType_Self-employed' 'EmploymentType_Unemployed'
 'MaritalStatus_Married' 'MaritalStatus_Single' 'LoanPurpose_Business'
 'LoanPurpose_Education' 'LoanPurpose_Home' 'LoanPurpose_Other']


In [33]:
# recreate engineered features used during model training

df["Income_to_Loan"] = (
    df["Income"] /
    (df["LoanAmount"] + 1)
)

df["Debt_burden"] = (
    df["LoanAmount"] /
    (df["Income"] + 1)
)

df["Score_pressure"] = (
    df["LoanAmount"] /
    (df["CreditScore"] + 1)
)

df["Risk_rate"] = (
    df["InterestRate"]
    *
    df["DTIRatio"]
)

df["Risk_score"] = (
    df["InterestRate"] * 2
    +
    df["DTIRatio"] * 2
)

df_model = df.drop("LoanID", axis=1)

X = df_model.drop("Default", axis=1)

X = pd.get_dummies(
    X,
    drop_first=True
)

X = X.reindex(
    columns=model.feature_names_in_,
    fill_value=0
)

df["Probability_of_Default"] = (
    model.predict_proba(X)[:, 1]
)

df["IncomeSegment"] = pd.qcut(
    df["Income"],
    q=4,
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)

In [37]:
print(df.columns.tolist())

['LoanID', 'Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio', 'Education', 'EmploymentType', 'MaritalStatus', 'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner', 'Default', 'Income_to_Loan', 'Debt_burden', 'Score_pressure', 'Risk_rate', 'Risk_score', 'Probability_of_Default', 'IncomeSegment']


In [35]:
print(
    df[
        [
            "Income",
            "LoanAmount",
            "Probability_of_Default",
            "IncomeSegment"
        ]
    ].head()
)

   Income  LoanAmount  Probability_of_Default IncomeSegment
0   85994       50587                0.036164          High
1   50432      124440                0.028946        Medium
2   84208      129188                0.164033          High
3   31713       44799                0.117799           Low
4   20437        9139                0.035242           Low


In [38]:
# borrowers who defaulted

defaulted = df[
    df["Default"] == 1
].copy()

print("Defaulted borrowers:")
print(len(defaulted))

Defaulted borrowers:
29653


In [39]:
term_summary = (
    defaulted["LoanTerm"]
    .value_counts()
    .sort_index()
)

print("\nNumber of defaulted borrowers by loan term")
print(term_summary)


Number of defaulted borrowers by loan term
LoanTerm
12    5920
24    5921
36    5907
48    5922
60    5983
Name: count, dtype: int64


In [40]:
term_pct = (
    defaulted["LoanTerm"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

print("\nPercentage distribution of defaulted borrowers by loan term")
print(term_pct.round(2))


Percentage distribution of defaulted borrowers by loan term
LoanTerm
12    19.96
24    19.97
36    19.92
48    19.97
60    20.18
Name: proportion, dtype: float64


In [41]:
portfolio_term_pct = (
    df["LoanTerm"]
    .value_counts(normalize=True)
    .sort_index()
    * 100
)

print("\nShare of total portfolio by loan term (%)")
print(portfolio_term_pct.round(2))


Share of total portfolio by loan term (%)
LoanTerm
12    19.96
24    19.98
36    20.00
48    20.04
60    20.03
Name: proportion, dtype: float64


In [42]:
defaulted.groupby("LoanTerm").agg(
    Borrowers=("LoanAmount", "count"),
    AvgIncome=("Income", "mean"),
    AvgLoan=("LoanAmount", "mean"),
    AvgCreditScore=("CreditScore", "mean")
).round(2)

,Borrowers,AvgIncome,AvgLoan,AvgCreditScore
LoanTerm,,,,
12,5920,71762.83,145100.69,557.80
24,5921,72344.06,144289.65,560.94
36,5907,71249.59,145529.05,560.64
48,5922,71782.35,143852.75,561.85
60,5983,72080.89,143814.37,555.25


In [43]:
print("\nDefault rate by Loan Term")

default_rate_by_term = (
    df
    .groupby("LoanTerm")["Default"]
    .mean()
    * 100
)

print(default_rate_by_term.round(2))


Default rate by Loan Term
LoanTerm
12    11.62
24    11.61
36    11.57
48    11.57
60    11.70
Name: Default, dtype: float64


In [44]:
df["IncomeSegment"] = pd.qcut(
    df["Income"],
    q=4,
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)



In [45]:
# How much loan exposure exists in the highest-risk income segment?
low_income = df[
    df["IncomeSegment"] == "Low"
]

print("\nLow-income borrower summary")

print(
    low_income[
        [
            "LoanAmount",
            "Probability_of_Default"
        ]
    ].describe()
)


Low-income borrower summary
          LoanAmount  Probability_of_Default
count   63837.000000            63837.000000
mean   127624.499084                0.176560
std     70711.929137                0.149720
min      5005.000000                0.004709
25%     66240.000000                0.065916
50%    127810.000000                0.129772
75%    188926.000000                0.242468
max    249993.000000                0.929314


In [46]:
risk_by_income = (
    df
    .groupby("IncomeSegment")
    .agg(
        Borrowers=("LoanID", "count"),
        AvgLoan=("LoanAmount", "mean"),
        AvgPD=("Probability_of_Default", "mean"),
        DefaultRate=("Default", "mean")
    )
)

risk_by_income["AvgPD"] *= 100
risk_by_income["DefaultRate"] *= 100

print(risk_by_income.round(2))

               Borrowers    AvgLoan  AvgPD  DefaultRate
IncomeSegment                                          
Low                63837  127624.50  17.66        17.38
Medium             63837  127713.95  11.08        10.54
High               63839  127344.35   9.50         9.51
Very High          63834  127632.67   8.78         9.02


In [47]:
# For Low Income borrowers, reduce loan amount by 10%.

simulation = low_income.copy()

simulation["NewLoanAmount"] = (
    simulation["LoanAmount"] * 0.90
)